# CISC7204 Suggested Exercise 05 — Practice 03 — Police Data

LU MENGJIE · MC643920

Questions follow the supplied Moodle exercise. Each code cell was executed against the supplied CSV. Large results show a preview, while the full filtered DataFrame is retained in the named variable.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

source = Path('../Datasets') / '3. Police Data.csv'
df = pd.read_csv(source)
print('Source:', source, '| rows:', len(df), '| columns:', len(df.columns))
display(df.head(3))

Source: ..\Datasets\3. Police Data.csv | rows: 65535 | columns: 15


,stop_date,stop_time,country_name,driver_gender,driver_age_raw,driver_age,driver_race,violation_raw,violation,search_conducted,search_type,stop_outcome,is_arrested,stop_duration,drugs_related_stop
0,1/2/2005,1:55,NaN,M,1985.0,20.0,White,Speeding,Speeding,False,NaN,Citation,False,0-15 Min,False
1,1/18/2005,8:15,NaN,M,1965.0,40.0,White,Speeding,Speeding,False,NaN,Citation,False,0-15 Min,False
2,1/23/2005,23:15,NaN,M,1972.0,33.0,White,Speeding,Speeding,False,NaN,Citation,False,0-15 Min,False


### Question 3.1: Remove columns containing only missing values

In [2]:
all_missing = df.columns[df.isna().all()].tolist()
df = df.drop(columns=all_missing)
print('Removed columns:', all_missing)
print('Remaining columns:', df.columns.tolist())

Removed columns: ['country_name']
Remaining columns: ['stop_date', 'stop_time', 'driver_gender', 'driver_age_raw', 'driver_age', 'driver_race', 'violation_raw', 'violation', 'search_conducted', 'search_type', 'stop_outcome', 'is_arrested', 'stop_duration', 'drugs_related_stop']


### Question 3.2: Who was stopped more often for speeding?

In [3]:
speeding = df[df['violation'].eq('Speeding')]
counts = speeding['driver_gender'].map({'M': 'Men', 'F': 'Women'}).value_counts()
display(counts.rename_axis('gender').to_frame('speeding_stops'))

,speeding_stops
gender,
Men,25517
Women,11686


### Question 3.3: Search rates by driver gender

In [4]:
known = df[df['driver_gender'].isin(['M', 'F'])].copy()
known['searched'] = known['search_conducted'].astype('boolean')
result = known.groupby('driver_gender').agg(stops=('searched','size'), searches=('searched','sum'), search_rate=('searched','mean'))
result.index = result.index.map({'M':'Men', 'F':'Women'})
display(result.assign(search_rate_percent=(result['search_rate']*100).round(2)))
print('This is an association; it does not establish that gender causes a search.')

,stops,searches,search_rate,search_rate_percent
driver_gender,,,,
Women,16310,366,0.02244,2.24
Men,45164,2113,0.046785,4.68


This is an association; it does not establish that gender causes a search.


### Question 3.4: Convert duration categories into approximate minutes and find average

In [5]:
duration_minutes = {'0-15 Min': 7.5, '16-30 Min': 23, '30+ Min': 45}
df['stop_duration_minutes_est'] = df['stop_duration'].map(duration_minutes)
print('Unmapped non-null categories:', sorted(set(df['stop_duration'].dropna()) - set(duration_minutes)))
print('Approximate mean stop duration (minutes):', df['stop_duration_minutes_est'].mean())
print('The 30+ category is represented by an assumed 45 minutes; the mean is an estimate.')

Unmapped non-null categories: ['2']
Approximate mean stop duration (minutes): 12.001195627419722
The 30+ category is represented by an assumed 45 minutes; the mean is an estimate.


### Question 3.5: Age distribution for each violation

In [6]:
display(df.groupby('violation')['driver_age'].describe().round(2))

,count,mean,std,min,25%,50%,75%,max
violation,,,,,,,,
Equipment,6507.0,31.68,11.38,16.0,23.0,28.0,39.0,81.0
Moving violation,11876.0,36.74,13.26,15.0,25.0,35.0,47.0,86.0
Other,3477.0,40.36,12.75,16.0,30.0,41.0,50.0,86.0
Registration/plates,2240.0,32.66,11.15,16.0,24.0,30.0,40.0,74.0
Seat belt,3.0,30.33,10.21,23.0,24.5,26.0,34.0,42.0
Speeding,37120.0,33.26,12.62,15.0,23.0,30.0,42.0,88.0
